In [1]:
from scipy.sparse.linalg import spsolve
import geopandas as gpd
import numpy as np
from libpysal import graph

In [4]:
sizes = [400,100,25]
rhos = np.arange(-0.9, 1.0, 0.1)
n_runs = 10
shapes = ["square","pent","hex"]

In [5]:
def simulate_autocorrelated_data(gdf, W, rhos, n_runs):    
    #SAR process
    n = len(gdf)
    I = identity(n)

    for rho in rhos:
        for run in range(n_runs):
            epsilon = np.random.normal(0, 1, n)
            y = spsolve(I - rho * W, epsilon)
            col_name = f"rho_{rho:.1f}_run_{run}"
            gdf[col_name] = y

In [6]:
for shape in shapes:
    for size in sizes:
        gdf = gpd.read_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet")
        g_true = graph.read_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
        simulate_autocorrelated_data(gdf,g_true.transform('R').sparse,rhos,n_runs)
        gdf.to_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet")